# Data Generation Demo

This notebook reads one case from `CT_RATE_demo_data`, exports NPZ files, inspects their structure, and creates a few training samples from the generated NPZ.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'paligemma_training_data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'training_file_generation'))
sys.path.insert(0, str(PROJECT_ROOT / 'paligemma_training_data'))
sys.path.insert(0, str(PROJECT_ROOT))

from generate_training_files import get_image_info
from paligemma_training_sample_generator import PaligemmaSampleGenerator

DEMO_ROOT = PROJECT_ROOT / 'CT_RATE_demo_data'
OUTPUT_DIR = PROJECT_ROOT / 'demo' / 'generated_npz_demo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PA_CASE_DIR = DEMO_ROOT / 'CT_RATE_projections_PA' / 'train_1_a_2'
LR_CASE_DIR = DEMO_ROOT / 'CT_RATE_projections_LR' / 'train_1_a_2'
PA_NPZ_PATH = OUTPUT_DIR / 'train_1_a_2_PA.npz'
LR_NPZ_PATH = OUTPUT_DIR / 'train_1_a_2_LR.npz'

print(f'Project root: {PROJECT_ROOT}')
print(f'PA case dir: {PA_CASE_DIR}')
print(f'LR case dir: {LR_CASE_DIR}')
print(f'NPZ output dir: {OUTPUT_DIR}')

In [ ]:
def list_case_pngs(case_dir):
    return sorted(path.name for path in case_dir.glob('*.png'))

pa_pngs = list_case_pngs(PA_CASE_DIR)
lr_pngs = list_case_pngs(LR_CASE_DIR)

print('PA case files:', pa_pngs[:10], '... total =', len(pa_pngs))
print('LR case files:', lr_pngs[:10], '... total =', len(lr_pngs))

In [ ]:
pa_info = get_image_info(str(PA_CASE_DIR), export_npz=True, npz_path=str(PA_NPZ_PATH))
lr_info = get_image_info(str(LR_CASE_DIR), export_npz=True, npz_path=str(LR_NPZ_PATH))

summary = {
    'PA': {'case': pa_info['case'], 'orientation': pa_info['orientation'], 'num_structures': pa_info['num_structures']},
    'LR': {'case': lr_info['case'], 'orientation': lr_info['orientation'], 'num_structures': lr_info['num_structures']},
}
print(json.dumps(summary, indent=2))

In [ ]:
loaded = np.load(PA_NPZ_PATH, allow_pickle=True)
metadata = loaded['metadata'].item()

print('NPZ arrays:', loaded.files)
print('Image shape:', loaded['img_array'].shape)
print('Metadata keys:', sorted(metadata.keys()))

structure_names = sorted(metadata['structure_info'].keys())
print('First 10 structures:', structure_names[:10])
sample_structure = structure_names[0]
print('Example structure metadata:')
print(json.dumps(metadata['structure_info'][sample_structure], indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(pa_info['img_array'], cmap='gray')
axes[0].set_title('PA ct.png')
axes[0].axis('off')

sample_mask_path = PA_CASE_DIR / f'{sample_structure}.png'
axes[1].imshow(np.array(Image.open(sample_mask_path).convert('L')), cmap='gray')
axes[1].set_title(f'Mask: {sample_structure}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
generator = PaligemmaSampleGenerator(str(PA_NPZ_PATH), enable_augmentation=False)
available_structures = generator.get_available_structures()
target_structure = 'heart' if 'heart' in available_structures else available_structures[0]

detection_sample = generator.generate_sample('detection', target_structure)
segmentation_sample = generator.generate_sample('segmentation', target_structure)
orientation_sample = generator.generate_sample('orientation_identification')

print('Detection prefix:', detection_sample.prefix)
print('Detection suffix:', detection_sample.suffix[:160])
print('Segmentation prefix:', segmentation_sample.prefix)
print('Segmentation suffix:', segmentation_sample.suffix[:160])
print('Orientation prompt:', orientation_sample.prefix)
print('Orientation answer:', orientation_sample.suffix)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.array(detection_sample.image), cmap='gray')
axes[0].set_title('Detection sample image')
axes[0].axis('off')

axes[1].imshow(np.array(segmentation_sample.image), cmap='gray')
axes[1].set_title('Segmentation sample image')
axes[1].axis('off')
plt.tight_layout()
plt.show()